In [18]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("movies_limpo.csv")

df.head()

,movie_id,title,content_type,genre_primary,genre_secondary,release_year,duration_minutes,rating,language,country_of_origin,imdb_rating,production_budget,box_office_revenue,number_of_seasons,number_of_episodes,is_netflix_original,added_to_platform,content_warning
0,movie_0001,Dragon Legend,Stand-up Comedy,History,Thriller,2014,35.0,TV-Y,French,Japan,6.4,NaN,NaN,0.0,0.0,False,2023-08-07,False
1,movie_0002,Storm Warrior,Stand-up Comedy,Sci-Fi,Nenhum,2017,37.0,PG,Japanese,USA,3.3,NaN,NaN,0.0,0.0,False,2022-01-28,True
2,movie_0003,Fire Family,Movie,Drama,Nenhum,2003,142.0,TV-MA,English,USA,8.5,2114120.0,NaN,0.0,0.0,False,2021-05-04,True
3,movie_0004,Our Princess,Documentary,Sci-Fi,Nenhum,2011,131.0,NC-17,Japanese,USA,5.3,NaN,NaN,0.0,0.0,False,2022-11-26,False
4,movie_0005,Warrior Mission,Documentary,Sport,Mystery,2015,91.0,TV-G,English,USA,3.1,NaN,NaN,0.0,0.0,False,2023-06-15,False


# Quem dá mais lucro, os Originais Netflix ou os Filmes Licenciados?

In [20]:
soma_lucro = df_financeiro.groupby("is_netflix_original")["lucro"].sum()
print(soma_lucro)

is_netflix_original
False    1.262602e+10
True     2.612392e+09
Name: lucro, dtype: float64


In [21]:
lucro_total = df_financeiro["lucro"].sum()
print(f"Lucro Total da Base: ${lucro_total:,.2f}")

Lucro Total da Base: $15,238,414,210.00


In [22]:
pct_licenciados = (soma_lucro[False] / lucro_total) * 100
print(f"A porcentagem de lucro entre as obras licenciadas e as obras originais resulta em {pct_licenciados:.2f}%")

A porcentagem de lucro entre as obras licenciadas e as obras originais resulta em 82.86%


### Embora a análise mostre que os filmes licenciados detêm 82% da soma de bilheteria global da base, é importante ressaltar que a Netflix opera por modelo de assinaturas e não recebe diretamente por bilheterias de terceiros. Portanto, a bilheteria aqui serve como um indicador (proxy) de popularidade e apelo comercial do conteúdo, e não de receita direta.

In [23]:
resumo_estatistico = df_financeiro.groupby('is_netflix_original')['lucro'].describe()

print(resumo_estatistico)

                     count          mean           std          min  \
is_netflix_original                                                   
False                192.0  6.576053e+07  2.129773e+08 -179391225.0   
True                  76.0  3.437358e+07  7.974296e+07 -191497233.0   

                           25%        50%         75%           max  
is_netflix_original                                                  
False               -2582194.5  5630039.5  34719020.0  2.025177e+09  
True                 -487525.5  8632437.5  37257431.0  3.885017e+08  


# Qual é o gênero mais aclamado pela crítica na Netflix, e quem ganha em qualidade: Originais ou Licenciados?

In [24]:
print(df['genre_primary'].value_counts())
print(df['imdb_rating'].describe())

genre_primary
Adventure      72
War            62
Sci-Fi         60
Comedy         60
Action         60
Animation      59
Western        55
History        54
Romance        54
Documentary    52
Biography      52
Crime          50
Horror         48
Family         48
Fantasy        47
Drama          45
Music          45
Sport          41
Mystery        39
Thriller       37
Name: count, dtype: int64
count    1040.00000
mean        6.28750
std         1.67405
min         0.50000
25%         5.50000
50%         6.40000
75%         7.20000
max        10.00000
Name: imdb_rating, dtype: float64


In [25]:
media_imdb = df["imdb_rating"].mean()
print(f"A media imdb resulta em {media_imdb}")

A media imdb resulta em 6.2875


In [26]:
imdb_por_genero = df.groupby(["genre_primary","content_type"])["imdb_rating"].mean().reset_index()
print(imdb_por_genero)

   genre_primary     content_type  imdb_rating
0         Action      Documentary     5.960000
1         Action   Limited Series     6.300000
2         Action            Movie     5.931818
3         Action  Stand-up Comedy     5.560000
4         Action        TV Series     6.607143
..           ...              ...          ...
93       Western      Documentary     6.300000
94       Western   Limited Series     6.400000
95       Western            Movie     6.484615
96       Western  Stand-up Comedy     6.000000
97       Western        TV Series     6.433333

[98 rows x 3 columns]


In [27]:
imdb_por_genero['distancia'] = (imdb_por_genero['imdb_rating'] - media_imdb).abs()
imdb_ordenado = imdb_por_genero.sort_values(by='distancia')
print(imdb_ordenado)

   genre_primary     content_type  imdb_rating  distancia
90           War            Movie     6.284000   0.003500
5      Adventure      Documentary     6.283333   0.004167
48       Fantasy        TV Series     6.281818   0.005682
76        Sci-Fi            Movie     6.280000   0.007500
47       Fantasy  Stand-up Comedy     6.300000   0.012500
..           ...              ...          ...        ...
11     Animation   Limited Series     5.275000   1.012500
31   Documentary   Limited Series     7.366667   1.079167
50       History   Limited Series     4.900000   1.387500
80         Sport   Limited Series     4.850000   1.437500
64       Mystery      Documentary     4.633333   1.654167

[98 rows x 4 columns]


### Se quisermos volume e consistência com risco baixo, devemos focar em Filmes de Guerra e Séries de Fantasia, que mantêm a nota padrão da plataforma. No entanto, o formato de Minisséries (Limited Series) é onde moram os extremos: documentários nesse formato são nossa maior aclamação ($7.36$), enquanto minisséries de história e esportes dão prejuízo de imagem, ficando abaixo de $5.0$.

### Agora iremos utilizar uma abordagem do Pandas muito importante: a correlação (.corr)

### Se o número for alto (ex: $> 0.5$): Significa que filmes com notas mais altas realmente dão mais lucro. A qualidade do conteúdo é a chave do sucesso financeiro.Se o número for baixo (ex: perto de $0.1$ ou $0.2$): Significa que a nota do IMDb não garante sucesso financeiro! Ou seja, um filme de nota $5.5$ pode dar tanto dinheiro quanto um de nota $8.0$.

In [28]:
df[['imdb_rating', 'production_budget', 'box_office_revenue', 'lucro']].corr()

,imdb_rating,production_budget,box_office_revenue,lucro
imdb_rating,1.000000,0.049245,-0.038172,-0.034869
production_budget,0.049245,1.000000,0.053841,-0.067260
box_office_revenue,-0.038172,0.053841,1.000000,0.992667
lucro,-0.034869,-0.067260,0.992667,1.000000


### A aclamação no IMDb é um mito financeiro. Nosso estudo provou que a nota do conteúdo não influencia o faturamento nem o lucro da plataforma ($r \approx 0$). Além disso, injetar mais orçamento em produções não aumenta a nota no IMDb nem o retorno financeiro. Conclusão: Devemos focar em produções de custo controlado que gerem volume bruto de audiência (faturamento), sem a obsessão de atingir notas altas de crítica.

In [29]:
# Salva o arquivo limpo e pronto para o Power BI
df.to_csv('netflix_catalog_limpo.csv', index=False)
print("Arquivo exportado com sucesso!")

Arquivo exportado com sucesso!


In [30]:
import os

# Mostra o caminho completo da pasta onde o Python salvou o arquivo
caminho_pasta = os.getcwd()
print(f"O seu arquivo está salvo nesta pasta:\n{caminho_pasta}")

# Confirma se o arquivo está realmente lá dentro
arquivo_existe = os.path.exists('netflix_catalog_limpo.csv')
print(f"\nO arquivo 'netflix_catalog_limpo.csv' foi encontrado? {arquivo_existe}")

O seu arquivo está salvo nesta pasta:
c:\Users\Ataide\.vscode

O arquivo 'netflix_catalog_limpo.csv' foi encontrado? True
